In [1]:
import pandas as pd
import plotly.graph_objects as go

In [2]:
# Custom functions
from utils.visualisation_functions import plot_sankey_hierarchy
from utils.data_manipulations import build_activity_name, add_site_id
from core.lci_database_builder import LCIDatabaseBuilder
from utils.conversion_functions import map_technosphere_to_ecoinvent, map_biosphere_to_ecoinvent
from utils.constants import CA_provinces
from utils.data_manipulations import add_land_substance_name

In [3]:
import bw2data as bd
from bw2io import BW2Package
import brightway2 as bw

# Plot selected sites

In [4]:
production_df = pd.read_excel(r'data/MetalliCan/sites_for_lci.xlsx', sheet_name='prod_data')

In [5]:
production_df_plot = production_df[production_df['Create_LCI?'] == 'Yes']
columns_to_plot = ['province', 'mining_processing_type', 'archetypes', 'Stream']

In [6]:
# plot_sankey_hierarchy(production_df_plot, columns_to_plot,
#                        html_output="data/MetalliCan/sites_for_lci_archetypes.html",
#                        output_image="data/MetalliCan/site_selection_sankey")

# Import cleaned MetalliCan data

In [7]:
# Market share
df_market = pd.read_excel(r'data/SI/SI_2_Site_selection.xlsx', sheet_name='market_shares')

In [8]:
# Pre-processed production table
production_df = pd.read_excel(r'data/MetalliCan/sites_for_lci.xlsx', sheet_name='prod_data')
production_df = production_df[production_df['Create_LCI?'] == 'Yes']

In [9]:
# Add activitiy_name to production_df and site_id
production_df['activity_name'] = production_df.apply(lambda row: build_activity_name(row, production_df), axis=1)
production_df = add_site_id(production_df)

In [10]:
# Normalized MetalliCan tables per ore processed
energy_stream_df = pd.read_csv(r'data/MetalliCan/data_for_lci_initialization/concentrate/energy_df.csv')
material_stream_df = pd.read_csv(r'data/MetalliCan/data_for_lci_initialization/concentrate/material_df.csv')
biosphere_stream_df = pd.read_csv(r'data/MetalliCan/data_for_lci_initialization/concentrate/biosphere_df.csv')
land_stream_df = pd.read_csv(r'data/MetalliCan/data_for_lci_initialization/concentrate/land_df.csv')
carbon_stock_stream_df = pd.read_csv(r'data/MetalliCan/data_for_lci_initialization/concentrate/carbon_stock_df.csv')

In [11]:
energy_stream_df_bgf = pd.read_csv(r'data/MetalliCan/data_for_lci_initialization/before_gap_filling/energy_df.csv')
material_stream_df_bgf = pd.read_csv(r'data/MetalliCan/data_for_lci_initialization/before_gap_filling/material_df.csv')
biosphere_stream_df_bgf = pd.read_csv(r'data/MetalliCan/data_for_lci_initialization/before_gap_filling/biosphere_df.csv')
land_stream_df_bgf = pd.read_csv(r'data/MetalliCan/data_for_lci_initialization/before_gap_filling/land_df.csv')
carbon_stock_stream_df_bgf = pd.read_csv(r'data/MetalliCan/data_for_lci_initialization/before_gap_filling/carbon_stock_df.csv')

In [12]:
# Only keep rows where site_id is in production_df
energy_stream_df = energy_stream_df[energy_stream_df['site_id'].isin(production_df['site_id'])]
material_stream_df = material_stream_df[material_stream_df['site_id'].isin(production_df['site_id'])]
biosphere_stream_df = biosphere_stream_df[biosphere_stream_df['site_id'].isin(production_df['site_id'])]
land_stream_df = land_stream_df[land_stream_df['site_id'].isin(production_df['site_id'])]
carbon_stock_stream_df = carbon_stock_stream_df[carbon_stock_stream_df['site_id'].isin(production_df['site_id'])]

energy_stream_df_bgf = energy_stream_df_bgf[energy_stream_df_bgf['site_id'].isin(production_df['site_id'])]
material_stream_df_bgf = material_stream_df_bgf[material_stream_df_bgf['site_id'].isin(production_df['site_id'])]
biosphere_stream_df_bgf = biosphere_stream_df_bgf[biosphere_stream_df_bgf['site_id'].isin(production_df['site_id'])]
land_stream_df_bgf = land_stream_df_bgf[land_stream_df_bgf['site_id'].isin(production_df['site_id'])]
carbon_stock_stream_df_bgf = carbon_stock_stream_df_bgf[carbon_stock_stream_df_bgf['site_id'].isin(production_df['site_id'])]

In [13]:
# Removing rows with value_normalized is NaN in the biosphere dfs
#biosphere_ore_df = biosphere_ore_df[~biosphere_ore_df['value_normalized'].isna()]
#biosphere_stream_df['value_normalized'] = biosphere_stream_df['value_normalized'] / 1e6

In [14]:
# Drop rows where 'unit' = 'ha'
biosphere_stream_df = biosphere_stream_df[biosphere_stream_df['unit'] != 'ha']
biosphere_stream_df_bgf = biosphere_stream_df_bgf[biosphere_stream_df_bgf['unit'] != 'ha']

In [15]:
land_stream_df = add_land_substance_name(land_stream_df)
land_stream_df_bgf = add_land_substance_name(land_stream_df_bgf)

In [16]:
carbon_stock_stream_df.rename(columns={"flow_type": "substance_name"}, inplace=True)
carbon_stock_stream_df_bgf.rename(columns={"flow_type": "substance_name"}, inplace=True)

# Keeping only relevant columns

In [17]:
nrj_col = ['activity_name', 'functional_unit', 'site_id', 'subflow_type', 'unit', 'value_normalized']
material_col = ['activity_name', 'functional_unit', 'site_id', 'subflow_type', 'unit', 'value_normalized']
biosphere_col = ['activity_name', 'functional_unit', 'site_id', 'substance_name', 'unit', 'value_normalized']

In [18]:
energy_stream_df = energy_stream_df[nrj_col]
material_stream_df = material_stream_df[material_col]
biosphere_stream_df = biosphere_stream_df[biosphere_col]
land_stream_df = land_stream_df[biosphere_col]

In [19]:
energy_stream_df_bgf = energy_stream_df_bgf[nrj_col]
material_stream_df_bgf = material_stream_df_bgf[material_col]
biosphere_stream_df_bgf = biosphere_stream_df_bgf[biosphere_col]
land_stream_df_bgf = land_stream_df_bgf[biosphere_col]

In [20]:
# Put energy_df and material_df together, and biosphere_df and land_df together
technosphere_stream_df = pd.concat([energy_stream_df, material_stream_df], ignore_index=True)
biosphere_stream_df = pd.concat([biosphere_stream_df, land_stream_df, carbon_stock_stream_df], ignore_index=True)
technosphere_stream_df_bgf = pd.concat([energy_stream_df_bgf, material_stream_df_bgf], ignore_index=True)
biosphere_stream_df_bgf = pd.concat([biosphere_stream_df_bgf, land_stream_df_bgf, carbon_stock_stream_df_bgf], ignore_index=True)

In [21]:
# Remove rows where value_normalized is NaN
technosphere_stream_df = technosphere_stream_df[~technosphere_stream_df['value_normalized'].isna()]
technosphere_stream_df_bgf = technosphere_stream_df_bgf[~technosphere_stream_df_bgf['value_normalized'].isna()]

In [22]:
# Add the province from the main_df to specify electricity location later
technosphere_stream_df = technosphere_stream_df.merge(production_df[['site_id', 'province']], on=['site_id'], how='left')
biosphere_stream_df = biosphere_stream_df.merge(production_df[['site_id', 'province']], on=['site_id'], how='left')
technosphere_stream_df_bgf = technosphere_stream_df_bgf.merge(production_df[['site_id', 'province']], on=['site_id'], how='left')
biosphere_stream_df_bgf = biosphere_stream_df_bgf.merge(production_df[['site_id', 'province']], on=['site_id'], how='left')

# Map MetalliCan flows to EI and RI flows

## Technosphere flows

In [23]:
mapping_technosphere_ri = pd.read_excel(r'data/Mappings/MAPPINGS_RI.xlsx', sheet_name='technosphere')
mapping_technosphere_ei = pd.read_excel(r'data/Mappings/MAPPINGS_EI.xlsx', sheet_name='technosphere')

In [24]:
# Apply the function
mapped_technosphere_ri_df = map_technosphere_to_ecoinvent(technosphere_stream_df, mapping_technosphere_ri, CA_provinces)
mapped_technosphere_ei_df = map_technosphere_to_ecoinvent(technosphere_stream_df, mapping_technosphere_ei, CA_provinces)

In [25]:
mapped_technosphere_ri_df_bgf = map_technosphere_to_ecoinvent(technosphere_stream_df_bgf, mapping_technosphere_ri, CA_provinces)
mapped_technosphere_ei_df_bgf = map_technosphere_to_ecoinvent(technosphere_stream_df_bgf, mapping_technosphere_ei, CA_provinces)

In [26]:
# Drop rows where ecoinvent_flow_name is "No mapping" and Amount is NaN for now
mapped_technosphere_ri_df = mapped_technosphere_ri_df[(mapped_technosphere_ri_df["Activity"] != "No mapping")]
mapped_technosphere_ei_df = mapped_technosphere_ei_df[(mapped_technosphere_ei_df["Activity"] != "No mapping")]
mapped_technosphere_ri_df_bgf = mapped_technosphere_ri_df_bgf[(mapped_technosphere_ri_df_bgf["Activity"] != "No mapping")]
mapped_technosphere_ei_df_bgf = mapped_technosphere_ei_df_bgf[(mapped_technosphere_ei_df_bgf["Activity"] != "No mapping")]

In [27]:
technosphere_col_for_lci = ['site_id', 'activity_name', 'functional_unit', 'Amount', 'Amount_min', 'Amount_mean', 'Amount_max', 'Activity', 'Product', 'Unit', 'Location', 'Database']

In [28]:
mapped_technosphere_ri_df = mapped_technosphere_ri_df[technosphere_col_for_lci]
mapped_technosphere_ei_df = mapped_technosphere_ei_df[technosphere_col_for_lci]
mapped_technosphere_ri_df_bgf = mapped_technosphere_ri_df_bgf[technosphere_col_for_lci]
mapped_technosphere_ei_df_bgf = mapped_technosphere_ei_df_bgf[technosphere_col_for_lci]

## Biosphere flows mapping

In [29]:
mapping_biosphere_ri = pd.read_excel(r'data/Mappings/MAPPINGS_RI.xlsx', sheet_name='biosphere')
mapping_biosphere_ei = pd.read_excel(r'data/Mappings/MAPPINGS_EI.xlsx', sheet_name='biosphere')

In [30]:
mapped_biosphere_ri_df = map_biosphere_to_ecoinvent(biosphere_stream_df, mapping_biosphere_ri, CA_provinces)
mapped_biosphere_ei_df = map_biosphere_to_ecoinvent(biosphere_stream_df, mapping_biosphere_ei, CA_provinces)
mapped_biosphere_ri_df_bgf = map_biosphere_to_ecoinvent(biosphere_stream_df_bgf, mapping_biosphere_ri, CA_provinces)
mapped_biosphere_ei_df_bgf = map_biosphere_to_ecoinvent(biosphere_stream_df_bgf, mapping_biosphere_ei, CA_provinces)

Index(['activity_name', 'functional_unit', 'site_id', 'substance_name', 'unit',
       'value_normalized', 'normalization_key', 'allocation_factor',
       'mining_processing_type', 'archetypes', 'data_source',
       'reference_mass_unit', 'province', 'Type', 'substance_id',
       'compartment_name', 'release_pathway', 'flow_direction',
       'MetalliCan_unit', 'DB_to_map', 'Flow name', 'Compartments', 'Unit',
       'Comment', 'Alternatives'],
      dtype='object')
⚠️ 1 biosphere flows could not be mapped to Ecoinvent:
   - PFCs
⚠️ No conversion defined for tco2eq → nan (flow: PFCs)
✅ Mapped 16249 biosphere flows (105 unique flows).
Index(['activity_name', 'functional_unit', 'site_id', 'substance_name', 'unit',
       'value_normalized', 'normalization_key', 'allocation_factor',
       'mining_processing_type', 'archetypes', 'data_source',
       'reference_mass_unit', 'province', 'Type', 'substance_id',
       'compartment_name', 'release_pathway', 'flow_direction',
       'Metall

In [31]:
# Drop rows where ecoinvent_flow_name is "No mapping" and Amount is NaN for now
mapped_biosphere_ri_df = mapped_biosphere_ri_df[(mapped_biosphere_ri_df["Flow Name"] != "No mapping") & (~mapped_biosphere_ri_df["Amount"].isna())]
mapped_biosphere_ei_df = mapped_biosphere_ei_df[(mapped_biosphere_ei_df["Flow Name"] != "No mapping") & (~mapped_biosphere_ei_df["Amount"].isna())]
mapped_biosphere_ri_df_bgf = mapped_biosphere_ri_df_bgf[(mapped_biosphere_ri_df_bgf["Flow Name"] != "No mapping") & (~mapped_biosphere_ri_df_bgf["Amount"].isna())]
mapped_biosphere_ei_df_bgf = mapped_biosphere_ei_df_bgf[(mapped_biosphere_ei_df_bgf["Flow Name"] != "No mapping") & (~mapped_biosphere_ei_df_bgf["Amount"].isna())]

In [32]:
biosphere_col_for_lci = ['site_id', 'activity_name', 'functional_unit', 'Amount', 'Unit', 'Flow Name', 'Compartments', 'Database']

In [33]:
mapped_biosphere_ri_df = mapped_biosphere_ri_df[biosphere_col_for_lci]
mapped_biosphere_ei_df = mapped_biosphere_ei_df[biosphere_col_for_lci]
mapped_biosphere_ri_df_bgf = mapped_biosphere_ri_df_bgf[biosphere_col_for_lci]
mapped_biosphere_ei_df_bgf = mapped_biosphere_ei_df_bgf[biosphere_col_for_lci]

In [34]:
# Add province and commodities from NRCan to production table
mapped_biosphere_ri_df = mapped_biosphere_ri_df.merge(production_df[['site_id', 'province', 'commodities']], on=['site_id'], how='left')
mapped_biosphere_ei_df = mapped_biosphere_ei_df.merge(production_df[['site_id', 'province', 'commodities']], on=['site_id'], how='left')
mapped_biosphere_ri_df_bgf = mapped_biosphere_ri_df_bgf.merge(production_df[['site_id', 'province', 'commodities']], on=['site_id'], how='left')
mapped_biosphere_ei_df_bgf = mapped_biosphere_ei_df_bgf.merge(production_df[['site_id', 'province', 'commodities']], on=['site_id'], how='left')

# LCI creation

## Regioinvent

In [35]:
# # Step 1 — initialize the builder
builder_regio = LCIDatabaseBuilder(db_name='metallican_lci_ri', project_name='metallican_new')

📂 Active Brightway project: metallican_new
✅ Using existing database 'metallican_lci_ri'.


In [36]:
# # Step 2 — create the activity shells from the main dataframe
builder_regio.build_lci_entries(df=mapped_biosphere_ri_df)
print(len(builder_regio.lcis))

✅ Created 45 base LCI activities with production exchanges.
45


In [37]:
# # Step 3a — Populate with the technosphere exchanges
builder_regio.populate_technosphere_exchanges(technosphere_df=mapped_technosphere_ri_df)

⚙️ Populating technosphere exchanges
   ✅ Cached 218246 activities from Regioinvent
   ✅ Cached 20769 activities from ecoinvent-3.10-cutoff regionalized
✅ Added 764 technosphere exchanges.


In [38]:
# # Step 3b — Populate with the biosphere exchanges
builder_regio.populate_biosphere_exchanges(biosphere_df=mapped_biosphere_ri_df)

🌱 Populating biosphere exchanges
   ✅ Cached 4362 biosphere flows from biosphere3
   ✅ Cached 110559 biosphere flows from biosphere3_spatialized_flows
✅ Added 11502 biosphere exchanges.


In [39]:
# # Step 4 - Consolidate duplicate flows
builder_regio.consolidate_exchanges()

🧮 Consolidation: 12311 → 1751 exchanges (summed duplicates).


In [40]:
builder_regio.build_market_activities(df_market)

🧩 Created 6 market activities.


6

In [41]:
builder_regio.write_to_database()

🧱 Writing 51 activities to database 'metallican_lci_ri'...


C:\Users\mp_ma\anaconda3\envs\lca\lib\site-packages\bw2data\backends\peewee\database.py:328: UserWarning: 
            Please use `del databases['metallican_lci_ri']` instead.
            Otherwise, the metadata and database get out of sync.
            Call `.delete(warn=False)` to skip this message in the future.
            
  warnings.warn(MESSAGE.format(self.name), UserWarning)


✅ Created 45 site-specific activities.
✅ Created 6 market activities.
✅ Saved exchanges: approx 1800 (failures: 0)
✅ Database 'metallican_lci_ri' processed successfully with 51 activities.


## Ecoinvent

In [42]:
# # Step 1 — initialize the builder
builder_ei = LCIDatabaseBuilder(db_name='metallican_lci_ei', project_name='metallican_new')

📂 Active Brightway project: metallican_new
✅ Using existing database 'metallican_lci_ei'.


In [43]:
# # Step 2 — create the activity shells from the main dataframe
builder_ei.build_lci_entries(df=mapped_biosphere_ri_df)
print(len(builder_ei.lcis))

✅ Created 45 base LCI activities with production exchanges.
45


In [44]:
# # Step 3a — Populate with the technosphere exchanges
builder_ei.populate_technosphere_exchanges(technosphere_df=mapped_technosphere_ei_df)

⚙️ Populating technosphere exchanges
   ✅ Cached 20769 activities from ecoinvent-3.10-cutoff
✅ Added 764 technosphere exchanges.


In [45]:
# # Step 3b — Populate with the biosphere exchanges
builder_ei.populate_biosphere_exchanges(biosphere_df=mapped_biosphere_ei_df)

🌱 Populating biosphere exchanges
   ✅ Cached 4362 biosphere flows from biosphere3
✅ Added 11502 biosphere exchanges.


In [46]:
# # Step 4 - Consolidate duplicate flows
builder_ei.consolidate_exchanges()

🧮 Consolidation: 12311 → 1716 exchanges (summed duplicates).


In [47]:
builder_ei.build_market_activities(df_market)

🧩 Created 6 market activities.


6

In [48]:
builder_ei.write_to_database()

🧱 Writing 51 activities to database 'metallican_lci_ei'...


C:\Users\mp_ma\anaconda3\envs\lca\lib\site-packages\bw2data\backends\peewee\database.py:328: UserWarning: 
            Please use `del databases['metallican_lci_ei']` instead.
            Otherwise, the metadata and database get out of sync.
            Call `.delete(warn=False)` to skip this message in the future.
            
  warnings.warn(MESSAGE.format(self.name), UserWarning)


✅ Created 45 site-specific activities.
✅ Created 6 market activities.
✅ Saved exchanges: approx 1765 (failures: 0)
✅ Database 'metallican_lci_ei' processed successfully with 51 activities.


In [49]:
import brightway2 as bw

db = bw.Database("metallican_lci_ri")
mkt = db.get("market_Cu_concentrate")   # ou ton code exact

rows = []
for exc in mkt.technosphere():
    rows.append({
        "supplier_code": exc.input.get("code"),
        "supplier_name": exc.input.get("name"),
        "supplier_location": exc.input.get("location"),
        "share": exc["amount"],
    })

rows = sorted(rows, key=lambda d: d["share"], reverse=True)

print("n inputs:", len(rows))
for r in rows[:20]:
    print(f"{r['share']:.6f} | {r['supplier_name']} | {r['supplier_location']} | {r['supplier_code']}")



n inputs: 10
0.368789 | Open-pit mining and beneficiation at Highland Valley | CA-BC | BC-MAIN-bf503b6b_Cu concentrate
0.200278 | Open-pit mining and beneficiation at Gibraltar | CA-BC | BC-MAIN-6b4800fe_Cu concentrate
0.101057 | Open-pit mining and beneficiation at Mount Milligan | CA-BC | BC-MAIN-ed23117f_Cu concentrate
0.081393 | Underground mining and beneficiation at Kidd Creek | CA-ON | ON-MAIN-f8313ebd_Cu concentrate
0.077432 | Underground mining and beneficiation at New Afton | CA-BC | BC-MAIN-aa76f6f2_Cu concentrate
0.068608 | Open-pit mining and beneficiation at Copper Mountain | CA-BC | BC-MAIN-599152a0_Cu concentrate
0.049245 | Open-pit mining and beneficiation at Mount Polley | CA-BC | BC-MAIN-3f490561_Cu concentrate
0.043772 | Underground mining and beneficiation at Snow Lake | CA-MB | GRP-a13779f8_Cu concentrate
0.009284 | Underground mining and beneficiation at LaRonde | CA-QC | QC-MAIN-e51eda66_Cu concentrate
0.000141 | Underground mining and beneficiation at Goldex | 

In [50]:
s = sum(exc["amount"] for exc in mkt.technosphere())
print("sum shares:", s)

sum shares: 0.9999999999999999


In [51]:
exc = list(mkt.technosphere())[0]
print(exc.as_dict())


{'output': ('metallican_lci_ri', 'market_Cu_concentrate'), 'input': ('metallican_lci_ri', 'BC-MAIN-599152a0_Cu concentrate'), 'amount': 0.06860778652421644, 'type': 'technosphere'}


# Before and after gap filling

### Before gap filling - technosphere only

In [52]:
# Step 1 — initialize the builder
builder_ri_bgf_tech = LCIDatabaseBuilder(db_name='metallican_bgf_tech', project_name='metallican_new')

📂 Active Brightway project: metallican_new
✅ Using existing database 'metallican_bgf_tech'.


In [53]:
# Step 2 — create the activity shells from the main dataframe
builder_ri_bgf_tech.build_lci_entries(df=mapped_biosphere_ri_df_bgf)
print(len(builder_ri_bgf_tech.lcis))

✅ Created 45 base LCI activities with production exchanges.
45


In [54]:
# Step 3a — Populate with the technosphere exchanges
builder_ri_bgf_tech.populate_technosphere_exchanges(technosphere_df=mapped_technosphere_ri_df_bgf)

⚙️ Populating technosphere exchanges
   ✅ Cached 218246 activities from Regioinvent
   ✅ Cached 20769 activities from ecoinvent-3.10-cutoff regionalized
✅ Added 227 technosphere exchanges.


In [55]:
# Step 3b — Populate with the biosphere exchanges
#builder_ei_bgf_tech.populate_biosphere_exchanges(biosphere_df=mapped_biosphere_ei_df)

In [56]:
# Step 4 - Consolidate duplicate flows
builder_ri_bgf_tech.consolidate_exchanges()

🧮 Consolidation: 272 → 209 exchanges (summed duplicates).


In [57]:
builder_ri_bgf_tech.write_to_database()

🧱 Writing 45 activities to database 'metallican_bgf_tech'...


C:\Users\mp_ma\anaconda3\envs\lca\lib\site-packages\bw2data\backends\peewee\database.py:328: UserWarning: 
            Please use `del databases['metallican_bgf_tech']` instead.
            Otherwise, the metadata and database get out of sync.
            Call `.delete(warn=False)` to skip this message in the future.
            
  warnings.warn(MESSAGE.format(self.name), UserWarning)


✅ Created 45 site-specific activities.
✅ Created 0 market activities.
✅ Saved exchanges: approx 209 (failures: 0)
✅ Database 'metallican_bgf_tech' processed successfully with 45 activities.


### Before gap filling - biosphere only

In [58]:
# Step 1 — initialize the builder
builder_ri_bgf_bio = LCIDatabaseBuilder(db_name='metallican_bgf_bio', project_name='metallican_new')

📂 Active Brightway project: metallican_new
✅ Using existing database 'metallican_bgf_bio'.


In [59]:
# Step 2 — create the activity shells from the main dataframe
builder_ri_bgf_bio.build_lci_entries(df=mapped_biosphere_ei_df_bgf)
print(len(builder_ri_bgf_tech.lcis))

✅ Created 45 base LCI activities with production exchanges.
45


In [60]:
# Step 3a — Populate with the technosphere exchanges
#builder_ei_bgf_bio.populate_technosphere_exchanges(technosphere_df=mapped_technosphere_ei_df)

In [61]:
# Step 3b — Populate with the biosphere exchanges
builder_ri_bgf_bio.populate_biosphere_exchanges(biosphere_df=mapped_biosphere_ei_df_bgf)

🌱 Populating biosphere exchanges
   ✅ Cached 4362 biosphere flows from biosphere3
✅ Added 11444 biosphere exchanges.


In [62]:
# Step 4 - Consolidate duplicate flows
builder_ri_bgf_bio.consolidate_exchanges()

🧮 Consolidation: 11489 → 1111 exchanges (summed duplicates).


In [63]:
builder_ri_bgf_bio.write_to_database()

🧱 Writing 45 activities to database 'metallican_bgf_bio'...


C:\Users\mp_ma\anaconda3\envs\lca\lib\site-packages\bw2data\backends\peewee\database.py:328: UserWarning: 
            Please use `del databases['metallican_bgf_bio']` instead.
            Otherwise, the metadata and database get out of sync.
            Call `.delete(warn=False)` to skip this message in the future.
            
  warnings.warn(MESSAGE.format(self.name), UserWarning)


✅ Created 45 site-specific activities.
✅ Created 0 market activities.
✅ Saved exchanges: approx 1111 (failures: 0)
✅ Database 'metallican_bgf_bio' processed successfully with 45 activities.


### After gap filling - technosphere only

In [64]:
# Step 1 — initialize the builder
builder_ri_agf_tech = LCIDatabaseBuilder(db_name='metallican_agf_tech', project_name='metallican_new')

📂 Active Brightway project: metallican_new
✅ Using existing database 'metallican_agf_tech'.


In [65]:
# Step 2 — create the activity shells from the main dataframe
builder_ri_agf_tech.build_lci_entries(df=mapped_biosphere_ei_df)
print(len(builder_ri_bgf_tech.lcis))

✅ Created 45 base LCI activities with production exchanges.
45


In [66]:
# Step 3a — Populate with the technosphere exchanges
builder_ri_agf_tech.populate_technosphere_exchanges(technosphere_df=mapped_technosphere_ei_df)

⚙️ Populating technosphere exchanges
   ✅ Cached 20769 activities from ecoinvent-3.10-cutoff
✅ Added 764 technosphere exchanges.


In [67]:
# Step 3b — Populate with the biosphere exchanges
#builder_ei_agf_tech.populate_biosphere_exchanges(biosphere_df=mapped_biosphere_ei_df)

In [68]:
# Step 4 - Consolidate duplicate flows
builder_ri_agf_tech.consolidate_exchanges()

🧮 Consolidation: 809 → 619 exchanges (summed duplicates).


In [69]:
builder_ri_agf_tech.write_to_database()

🧱 Writing 45 activities to database 'metallican_agf_tech'...


C:\Users\mp_ma\anaconda3\envs\lca\lib\site-packages\bw2data\backends\peewee\database.py:328: UserWarning: 
            Please use `del databases['metallican_agf_tech']` instead.
            Otherwise, the metadata and database get out of sync.
            Call `.delete(warn=False)` to skip this message in the future.
            
  warnings.warn(MESSAGE.format(self.name), UserWarning)


✅ Created 45 site-specific activities.
✅ Created 0 market activities.
✅ Saved exchanges: approx 619 (failures: 0)
✅ Database 'metallican_agf_tech' processed successfully with 45 activities.


### After gap filling - biosphere only

In [70]:
# Step 1 — initialize the builder
builder_ri_bgf_bio = LCIDatabaseBuilder(db_name='metallican_agf_bio', project_name='metallican_new')

📂 Active Brightway project: metallican_new
✅ Using existing database 'metallican_agf_bio'.


In [71]:
# Step 2 — create the activity shells from the main dataframe
builder_ri_bgf_bio.build_lci_entries(df=mapped_biosphere_ei_df)
print(len(builder_ri_bgf_tech.lcis))

✅ Created 45 base LCI activities with production exchanges.
45


In [72]:
# Step 3a — Populate with the technosphere exchanges
#builder_ei_bgf_bio.populate_technosphere_exchanges(technosphere_df=mapped_technosphere_ei_df)

In [73]:
# Step 3b — Populate with the biosphere exchanges
builder_ri_bgf_bio.populate_biosphere_exchanges(biosphere_df=mapped_biosphere_ei_df)

🌱 Populating biosphere exchanges
   ✅ Cached 4362 biosphere flows from biosphere3
✅ Added 11502 biosphere exchanges.


In [74]:
# Step 4 - Consolidate duplicate flows
builder_ri_bgf_bio.consolidate_exchanges()

🧮 Consolidation: 11547 → 1142 exchanges (summed duplicates).


In [75]:
builder_ri_bgf_bio.write_to_database()

🧱 Writing 45 activities to database 'metallican_agf_bio'...


C:\Users\mp_ma\anaconda3\envs\lca\lib\site-packages\bw2data\backends\peewee\database.py:328: UserWarning: 
            Please use `del databases['metallican_agf_bio']` instead.
            Otherwise, the metadata and database get out of sync.
            Call `.delete(warn=False)` to skip this message in the future.
            
  warnings.warn(MESSAGE.format(self.name), UserWarning)


✅ Created 45 site-specific activities.
✅ Created 0 market activities.
✅ Saved exchanges: approx 1142 (failures: 0)
✅ Database 'metallican_agf_bio' processed successfully with 45 activities.


# Exports databases

In [76]:
from core.lci_database_builder import export_bw_database_to_excel

In [77]:
bd.projects.set_current("metallican_new")

In [78]:
fp_ei = BW2Package.export_obj(
    bd.Database("metallican_lci_ei"),
    filename="metallican_lci_ei",
    folder=r"C:\Users\mp_ma\OneDrive - polymtl\POST_DOC\CODE\regionalized_lci_mineral\exported_bw2_databases",
)
print("Exported to:", fp_ei)

Exported to: C:\Users\mp_ma\OneDrive - polymtl\POST_DOC\CODE\regionalized_lci_mineral\exported_bw2_databases\metallican_lci_ei.22c678a104a09ee0f51ac9ef9c9bfd21.bw2package


In [79]:
fp_ri = BW2Package.export_obj(
    bd.Database("metallican_lci_ri"),
    filename="metallican_lci_ri",
    folder=r"C:\Users\mp_ma\OneDrive - polymtl\POST_DOC\CODE\regionalized_lci_mineral\exported_bw2_databases",
)
print("Exported to:", fp_ri)

Exported to: C:\Users\mp_ma\OneDrive - polymtl\POST_DOC\CODE\regionalized_lci_mineral\exported_bw2_databases\metallican_lci_ri.d99527f2f15c54371f8254f102321b4c.bw2package
